# Notebook 06: Mixed Precision Training

## Why This Matters

Mixed precision is not a nice-to-have optimisation -- it is table stakes for any serious deep learning work. On modern NVIDIA hardware (Ampere, Hopper), BF16 matmuls run on Tensor Cores at 2-4x the throughput of FP32, while using half the memory. This directly translates to either larger batch sizes (better gradient estimates), larger models (better capacity), or faster iterations (more experiments per day). Getting mixed precision wrong -- applying it to ops that need full precision, or using FP16 without a loss scaler -- causes silent accuracy degradation or NaN gradients.

## Background

Floating-point formats differ in their tradeoffs between range and precision:

| Format | Sign | Exponent | Mantissa | Range | Precision |
|--------|------|----------|----------|-------|----------|
| FP32 | 1 | 8 | 23 | ~1e-38 to ~3e38 | ~7 decimal digits |
| FP16 | 1 | 5 | 10 | ~6e-5 to 65504 | ~3 decimal digits |
| BF16 | 1 | 8 | 7 | ~1e-38 to ~3e38 | ~2 decimal digits |

The key insight: **BF16 has the same exponent width as FP32** (same range) but reduced mantissa (less precision). FP16 has higher precision than BF16 but much smaller range, making it prone to overflow/underflow without careful loss scaling. On Ampere+ GPUs and Apple MPS, BF16 is almost always the right choice over FP16.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

# Check available formats
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"BF16 supported (CUDA): {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'N/A'}")
print(f"MPS available:  {torch.backends.mps.is_available()}")
print("Ready.")

## 1. FP32 vs FP16 vs BF16 -- Dynamic Range vs Precision

The practical consequence of FP16's limited range (max ~65504) is that gradient norms that exceed this threshold become `inf`, propagating NaN through the network. This is why FP16 training requires a GradScaler. BF16 sidesteps the issue entirely: the same exponent range as FP32 means gradient values that are valid in FP32 are valid in BF16.

In [ ]:
# Demonstrate overflow/underflow differences
vals = [0.0001, 0.001, 1.0, 1000.0, 65000.0, 70000.0, 1e8, 1e38]

print(f"{'Value':>12} | {'FP32':>12} | {'FP16':>12} | {'BF16':>12}")
print("-" * 55)
for v in vals:
    t = torch.tensor(v)
    fp32 = t.to(torch.float32).item()
    fp16 = t.to(torch.float16).item()
    bf16 = t.to(torch.bfloat16).item()
    print(f"{v:>12.4g} | {fp32:>12.4g} | {fp16:>12.4g} | {bf16:>12.4g}")

print()
# Precision comparison at small values
print("Precision near 1.0:")
for delta in [0.1, 0.01, 0.001, 0.0001]:
    v = 1.0 + delta
    fp16 = torch.tensor(v).to(torch.float16).item()
    bf16 = torch.tensor(v).to(torch.bfloat16).item()
    fp32 = torch.tensor(v).to(torch.float32).item()
    print(f"  1 + {delta}: fp32={fp32:.6f}  fp16={fp16:.6f}  bf16={bf16:.6f}")

## 2. torch.amp.autocast -- Automatic Precision Selection

The `autocast` context manager automatically selects the optimal dtype for each operation. Operations that benefit from reduced precision (matmul, conv) run in the specified low-precision dtype. Operations that require full precision for numerical stability (softmax, batch norm statistics, loss computation) run in FP32 -- autocast handles this automatically, not the user.

You do NOT need to cast your model parameters or inputs -- autocast does it internally. Your model stays in FP32; autocast only affects kernel dispatch.

In [ ]:
model = nn.Sequential(
    nn.Linear(128, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

x = torch.randn(32, 128, device=device)

# Determine which dtype to use for autocast
if device == "cuda":
    amp_dtype = torch.bfloat16  # Ampere+ preferred
elif device == "mps":
    amp_dtype = torch.bfloat16  # MPS supports BF16
else:
    amp_dtype = torch.float16   # CPU FP16 (for demo; rarely useful in practice)

print(f"Using autocast dtype: {amp_dtype}")
print(f"Model parameters dtype (unchanged): {next(model.parameters()).dtype}")

# Without autocast
with torch.no_grad():
    out_fp32 = model(x)
    print(f"Without autocast: input {x.dtype}, output {out_fp32.dtype}")

# With autocast
with torch.no_grad():
    with torch.amp.autocast(device_type=device if device != "mps" else "cpu", dtype=amp_dtype):
        out_amp = model(x)
        print(f"With autocast:    input {x.dtype}, output {out_amp.dtype}")

print(f"\nOutput close (FP32 vs AMP): {torch.allclose(out_fp32, out_amp.float(), atol=1e-2)}")

## 3. GradScaler for FP16 -- Loss Scaling

FP16 gradients can underflow to zero for small values that would be representable in FP32. Loss scaling prevents this by multiplying the loss by a large scalar (e.g., 2^16) before `backward()`, shifting gradient values into the representable FP16 range. Before the optimizer step, `scaler.unscale_()` divides gradients back by the scale factor. If any gradient is `inf` or `nan` after unscaling, the step is skipped and the scale factor is halved.

**BF16 does not need a GradScaler** -- it has the same dynamic range as FP32. GradScaler is FP16-only.

In [ ]:
# GradScaler is only needed for FP16 on CUDA
# For demonstration we simulate the pattern

if torch.cuda.is_available():
    demo_device = "cuda"
    use_fp16 = True
else:
    demo_device = device
    use_fp16 = False  # use BF16 pattern instead

model_amp = nn.Sequential(
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10)
).to(demo_device)

optimizer_amp = torch.optim.AdamW(model_amp.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(enabled=use_fp16)  # no-op when enabled=False

X_amp = torch.randn(32, 64, device=demo_device)
y_amp = torch.randint(0, 10, (32,), device=demo_device)

# Full FP16 / BF16 training step
for step in range(3):
    optimizer_amp.zero_grad(set_to_none=True)

    autocast_dtype = torch.float16 if use_fp16 else torch.bfloat16
    autocast_device = "cuda" if demo_device == "cuda" else "cpu"

    with torch.amp.autocast(device_type=autocast_device, dtype=autocast_dtype):
        logits = model_amp(X_amp)
        loss = criterion(logits, y_amp)

    # Scale loss, backward
    scaler.scale(loss).backward()

    # CRITICAL: unscale_ BEFORE gradient clipping
    scaler.unscale_(optimizer_amp)
    nn.utils.clip_grad_norm_(model_amp.parameters(), max_norm=1.0)

    # Step: skip if grads contain inf/nan
    scaler.step(optimizer_amp)
    scaler.update()

    print(f"step {step}: loss={loss.item():.4f}  scale={scaler.get_scale():.0f}")

print(f"\nFP16 GradScaler enabled: {use_fp16}")
print("Pattern: autocast -> scale(loss).backward() -> unscale_ -> clip -> step -> update")

## 4. The Correct Mixed Precision Training Loop

Putting everything together: the production-ready FP16/BF16 training step. Notice the ordering of operations -- especially `unscale_` before clip and `update` after step.

In [ ]:
def make_model_and_opt(input_dim=64, hidden=256, output_dim=10, device="cpu"):
    model = nn.Sequential(
        nn.Linear(input_dim, hidden), nn.GELU(),
        nn.Linear(hidden, hidden), nn.GELU(),
        nn.Linear(hidden, output_dim),
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    return model, opt

# Determine best AMP config for this machine
if torch.cuda.is_available():
    amp_device_type = "cuda"
    amp_dtype = torch.bfloat16
    use_scaler = False   # BF16 doesn't need scaler on CUDA
elif torch.backends.mps.is_available():
    amp_device_type = "cpu"  # MPS autocast uses CPU path
    amp_dtype = torch.bfloat16
    use_scaler = False
else:
    amp_device_type = "cpu"
    amp_dtype = torch.float32  # CPU: no speedup from AMP; skip
    use_scaler = False

model_prod, opt_prod = make_model_and_opt(device=device)
scaler_prod = torch.amp.GradScaler(enabled=use_scaler)
criterion = nn.CrossEntropyLoss()

X = torch.randn(64, 64, device=device)
y = torch.randint(0, 10, (64,), device=device)

losses = []
for step in range(10):
    opt_prod.zero_grad(set_to_none=True)

    with torch.amp.autocast(device_type=amp_device_type, dtype=amp_dtype):
        logits = model_prod(X)
        loss = criterion(logits, y)

    scaler_prod.scale(loss).backward()

    if use_scaler:
        scaler_prod.unscale_(opt_prod)

    nn.utils.clip_grad_norm_(model_prod.parameters(), max_norm=1.0)

    scaler_prod.step(opt_prod)
    scaler_prod.update()
    losses.append(loss.item())

print("Loss trajectory:", [f"{l:.3f}" for l in losses])
print(f"AMP config: device_type={amp_device_type}, dtype={amp_dtype}, scaler={use_scaler}")

## 5. Common Pitfalls

Three patterns that cause silent failures in mixed precision training.

In [ ]:
# Pitfall 1: Applying log_softmax in autocast then NLLLoss outside -- double cast
# This is why you should prefer CrossEntropyLoss (handles everything internally)
x_logit = torch.randn(4, 5, device=device, dtype=torch.float16)

# Wrong: log_softmax in FP16, then NLLLoss
try:
    log_prob = torch.nn.functional.log_softmax(x_logit, dim=-1)
    target = torch.randint(0, 5, (4,), device=device)
    loss_bad = torch.nn.functional.nll_loss(log_prob, target)
    print(f"NLLLoss in FP16: {loss_bad.item():.4f} (may be nan with extreme inputs)")
except Exception as e:
    print(f"Error: {e}")

# Correct: CrossEntropyLoss handles casting internally
loss_good = torch.nn.functional.cross_entropy(x_logit.float(), target)
print(f"CrossEntropyLoss in FP32 cast: {loss_good.item():.4f}")

# Pitfall 2: Comparing tensors of different dtypes in assertions
t_fp32 = torch.tensor([1.0, 2.0])
t_fp16 = t_fp32.half()
print(f"\nFP16 round-trip error: {(t_fp32 - t_fp16.float()).abs().max().item()}")
print("Always cast to FP32 before numerical comparisons in tests")

# Pitfall 3: Model outputs still in low precision after autocast
model_small = nn.Linear(4, 2).to(device)
x_in = torch.randn(2, 4, device=device)
with torch.amp.autocast(device_type=amp_device_type, dtype=torch.float16 if not torch.backends.mps.is_available() else torch.bfloat16):
    out = model_small(x_in)
print(f"\nModel output dtype inside autocast: {out.dtype}")
print("Always .float() outputs before custom loss functions that need FP32")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| BF16 vs FP16 | BF16 has same range as FP32; FP16 has higher precision but overflows easily |
| autocast | Automatically casts ops to low precision; model stays in FP32 |
| GradScaler | FP16 only; scales loss to prevent gradient underflow |
| unscale_ before clip | Clip operates on real-scale gradients, not inflated ones |
| scaler.update() | Halves scale factor on inf/nan; call every step |
| BF16 no scaler | BF16 range = FP32 range; no underflow; GradScaler disabled |

### Common Pitfalls
- Using GradScaler with BF16 -- unnecessary but also harmless (enabled=False by default)
- Applying softmax + NLLLoss instead of CrossEntropyLoss inside autocast -- NaN-prone
- Clipping before `unscale_()` -- clips scaled gradients, not real gradients
- Forgetting `scaler.update()` -- scale never adapts, stays at initial value
- Using FP16 on older hardware (pre-Volta) -- no Tensor Cores benefit, just precision loss

### Next: Notebook 07 -- ResNet from Scratch